# 04. RL fine-tuning

Stage 4 of `docs/codeagents/research_plan.md` (§6.3, §7, §11 stage 4): the base model (`runs/base/best.pt`, π_ref) is fine-tuned with RLOO + exact KL to π_ref against the proxy RM (`runs/rm_proxy/best.pt`); the gold RM (`runs/rm_gold/best.pt`) is used for evaluation only. Run `02_baseline` and `03_reward_model` first.

- `rl_ratio_b0.1`: R1, reward = E[played ratio], β = 0.1. Expected hacking (plan §12): shorter, more popular and repeated tracks, proxy reward growing faster than gold.
- `rl_debiased_b0.1`: R4, ratio minus the train mean ratio of the track-length bin, which removes the advantage of short tracks.

Then, with R1 and β = 0.1 fixed, the objectives on the same G samples and rewards are compared (`rl.method`): RLOO, GRPO, Best-of-N fine-tuning (cross-entropy on the best sample by the RM) and online DPO (best vs worst sample of the group, `kl_beta` is the DPO β).

Stage 5 (after the first results, `docs/codeagents/observations.md` §5): a β sweep for R1 and R4 below 0.1, where the interesting KL range is, and GRPO / BoN / DPO rerun with their own β so that the methods are compared at a similar KL, not at the same nominal β.

Every `rl.eval_every` steps (and at step 0 = π_ref) the policy is evaluated on the val day: NDCG / Recall, proxy and gold reward of the top-10, its composition (share of history tracks, mean length, popularity, Gini, distinct artists) and the KL to π_ref. Logic lives in `recsys_rl.rl` and `recsys_rl.rewards`; the same run from a terminal: `uv run yambda-train-rl run_name=rl_ratio_b0.1`. A run resumes from `runs/<run_name>/last.pt` after a disconnect. Tables and plots for the report go to `results/stage4/` on Drive.

In [ ]:
# Bootstrap: each Colab notebook runs in its own runtime, so repeat the setup from 00.
# Data must already be on Drive (00_colab_setup.ipynb downloads it). Locally: cd to repo root.
import os
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    os.environ["RECSYS_DATA_ROOT"] = "/content/drive/MyDrive/tlab-recsys"
    token = (userdata.get("GH_TOKEN") or "").strip()
    if not token:
        raise RuntimeError("Colab secret GH_TOKEN is empty, see notebooks/README.md")
    repo_url = f"https://{token}@github.com/miont/tlab-2026-recsys-task2.git"
    !git clone -q {repo_url} /content/repo || git -C /content/repo pull -q
    if not Path("/content/repo/pyproject.toml").exists():
        raise RuntimeError("git clone failed: check that GH_TOKEN can read the repository")
    %cd /content/repo
    !pip install -q uv
    !uv pip install --system -q -e .
    sys.path.insert(0, "/content/repo/src")  # the editable .pth is only read at startup
elif not Path("configs").exists():
    os.chdir(Path.cwd().parent)

In [ ]:
import matplotlib.pyplot as plt
import polars as pl
import torch

from recsys_rl import evaluate
from recsys_rl.config import load_config, parse_overrides
from recsys_rl.data import prepare
from recsys_rl.rl import train_rl
from recsys_rl.train import amp_dtype
from recsys_rl.utils import get_device, save_figure

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(250)
pl.Config.set_float_precision(4)

device = get_device()
print(device, torch.cuda.get_device_name() if device.type == "cuda" else "")
cfg = load_config("configs/base.yaml")
prepared = prepare(cfg)
RESULTS = cfg.results_dir / "stage4"  # CSV / PNG for the report, on Drive in Colab
RESULTS.mkdir(parents=True, exist_ok=True)

## Training

Defaults are in `RLConfig` (`src/recsys_rl/config.py`): 3000 steps of 64 contexts × 16 sampled candidates, lr 5e-5, evaluation every 250 steps. Overrides use the CLI syntax, e.g. `"rl.kl_beta=0.05"`.

In [ ]:
RUNS = {
    "rl_ratio_b0.1": ["rl.reward=ratio", "rl.kl_beta=0.1"],
    "rl_debiased_b0.1": ["rl.reward=debiased_ratio", "rl.kl_beta=0.1"],
}


def run(run_name: str, *overrides: str):
    run_cfg = load_config(
        "configs/base.yaml", **parse_overrides([f"run_name={run_name}", *overrides])
    )
    return train_rl(run_cfg, prepared, device)  # path to policy.pt


policies = {name: run(name, *overrides) for name, overrides in RUNS.items()}  # name -> path

## Trajectories on the val day

Step 0 is π_ref. Proxy reward growing while gold reward stalls or falls is the over-optimisation signal (Gao et al., 2022); the composition panels show what the policy moves towards.

In [ ]:
PANELS = [
    ("val_proxy_reward@10", "proxy reward@10"),
    ("val_gold_reward@10", "gold reward@10"),
    ("val_ndcg@10", "NDCG@10"),
    ("val_recall@100", "Recall@100"),
    ("val_kl_ref", "KL(π || π_ref)"),
    ("val_repeat@10", "share of history tracks@10"),
    ("val_length@10", "mean track length@10, s"),
    ("val_log_count@10", "mean log10 train count@10"),
    ("val_coverage@10", "coverage@10"),
    ("val_artists@10", "distinct artists@10"),  # needs artist_item_mapping.parquet
]


def plot_trajectories(runs: list[str], name: str) -> pl.DataFrame:
    logs = pl.concat(
        [
            pl.read_ndjson(cfg.run_dir.parent / run / "log.jsonl").with_columns(run=pl.lit(run))
            for run in runs
        ],
        how="diagonal_relaxed",
    )
    logs.write_csv(RESULTS / f"{name}.csv")
    panels = [(col, title) for col, title in PANELS if col in logs.columns]
    fig, axes = plt.subplots(2, 5, figsize=(25, 8))
    for ax, (col, title) in zip(axes.ravel(), panels, strict=False):  # artists may be absent
        for run in runs:
            log = logs.filter(pl.col("run") == run)
            ax.plot(log["step"], log[col], marker="o", label=run)
        ax.set(xlabel="step", title=title)
    for ax in axes.ravel()[len(panels) :]:
        ax.axis("off")
    axes[0, 0].legend()
    save_figure(fig, RESULTS / f"{name}.png")
    plt.show()
    return logs.select("run", "step", *[col for col, _ in panels], "train_reward", "train_entropy")


plot_trajectories(list(RUNS), "trajectories")

## Objectives on the same samples (R1, β = 0.1)

All runs sample G = 16 candidates per context and score them with the same proxy RM; only the update differs. `bon` is the training-time counterpart of reranking the policy's candidates by the RM; `dpo` takes the best and the worst sample of each group as a preference pair (pairs from RM-ranked policy samples, as in OneRec) and has no explicit KL term, its `kl_beta` is the DPO β. `train_pair_share` in the log is the share of groups that gave a pair.

In [ ]:
METHOD_RUNS = {
    f"rl_ratio_{method}_b0.1": ["rl.reward=ratio", "rl.kl_beta=0.1", f"rl.method={method}"]
    for method in ("grpo", "bon", "dpo")
}
policies |= {name: run(name, *overrides) for name, overrides in METHOD_RUNS.items()}
plot_trajectories(["rl_ratio_b0.1", *METHOD_RUNS], "methods")

## β sweep (R1 and R4)

At β = 0.1 RLOO reaches its KL-regularised optimum: within a group the reward std is ≈ 0.043, and for π* ∝ π_ref·exp(r/β) the expected KL is ≈ σ²/(2β²) ≈ 0.09 (observed 0.11–0.14), with NDCG unchanged. GRPO at the same β collapsed (KL ≈ 6) because dividing the advantage by σ makes its effective β ≈ 0.1·σ ≈ 0.004 in RLOO units. So the trade-off lies between these two points; the grid of the plan (up to β = 1) would give KL ≈ 0. Rough KL predictions: β = 0.05 → 0.4, 0.02 → 2, smaller β beyond the small-tilt approximation.

The first 500 steps are evaluated every 50 steps (`rl.eval_early_every`), because with small β the policy moves away from π_ref early.

In [ ]:
BETAS = (0.05, 0.02, 0.01, 0.005)
EARLY = ["rl.eval_early_every=50", "rl.eval_early_until=500"]
SWEEP_RUNS = {
    f"rl_{name}_b{beta}": [f"rl.reward={reward}", f"rl.kl_beta={beta}", *EARLY]
    for name, reward in (("ratio", "ratio"), ("debiased", "debiased_ratio"))
    for beta in BETAS
}
policies |= {name: run(name, *overrides) for name, overrides in SWEEP_RUNS.items()}
plot_trajectories(["rl_ratio_b0.1", *[f"rl_ratio_b{b}" for b in BETAS]], "sweep_ratio")
plot_trajectories(["rl_debiased_b0.1", *[f"rl_debiased_b{b}" for b in BETAS]], "sweep_debiased")

## Objectives at matched β

GRPO divides the advantage by σ ≈ 0.04, BoN's cross-entropy on the best sample is not centred at all, and DPO at β = 0.1 only saturates at a log-ratio margin of ≈ 30, so at β = 0.1 all three drifted to KL 5–7 while RLOO stayed at 0.14. The reruns use β scaled to the reward: GRPO β = 2.5 ≈ RLOO β = 0.1 and β = 0.5 ≈ RLOO β = 0.02; `grpo_std=batch` divides by one std of the whole batch instead of the per-group one (groups with near-equal rewards, mostly RM noise, are no longer amplified); BoN and DPO at β = 1 and 3. All runs should be compared on the KL axis below, not by β.

In [ ]:
MATCHED_RUNS = {
    "rl_ratio_grpo_b2.5": ["rl.method=grpo", "rl.kl_beta=2.5"],
    "rl_ratio_grpo_b0.5": ["rl.method=grpo", "rl.kl_beta=0.5"],
    "rl_ratio_grpo_batch_b0.5": ["rl.method=grpo", "rl.grpo_std=batch", "rl.kl_beta=0.5"],
    "rl_ratio_bon_b1": ["rl.method=bon", "rl.kl_beta=1"],
    "rl_ratio_bon_b3": ["rl.method=bon", "rl.kl_beta=3"],
    "rl_ratio_dpo_b1": ["rl.method=dpo", "rl.kl_beta=1"],
    "rl_ratio_dpo_b3": ["rl.method=dpo", "rl.kl_beta=3"],
}
MATCHED_RUNS = {
    name: ["rl.reward=ratio", *overrides, *EARLY] for name, overrides in MATCHED_RUNS.items()
}
policies |= {name: run(name, *overrides) for name, overrides in MATCHED_RUNS.items()}
plot_trajectories(["rl_ratio_b0.1", "rl_ratio_b0.02", *MATCHED_RUNS], "methods_matched")

## Reranking baseline

The KL-regularised optimum π* ∝ π_ref · exp(r / β) computed at inference (`recsys_rl.rerank`): the proxy RM scores the top-100 of π_ref with the run's reward, items outside the top-100 keep the π_ref-weighted mean reward, so KL(π_β || π_ref) is exact and the points share the KL axis with the RL runs. β = ∞ is π_ref, β = 0 the greedy rerank of the top-100 by the reward alone (no KL). If the reranker reaches the RL curve, RL only amortises the policy + RM pair into one model.

In [ ]:
from recsys_rl.rerank import rerank_eval

RERANK_RUNS = {"rerank_ratio": "ratio", "rerank_debiased": "debiased_ratio"}
rerank = {
    split: pl.concat(
        [
            rerank_eval(
                load_config("configs/base.yaml", **parse_overrides([f"rl.reward={reward}"])),
                prepared,
                device,
                split=split,
                name=name,
            )
            for name, reward in RERANK_RUNS.items()
        ]
    )
    for split in ("val", "test")
}
for split, table in rerank.items():
    table.write_csv(RESULTS / f"rerank_{split}.csv")
rerank["val"].select(
    "run",
    "beta",
    "val_ndcg@10",
    "val_kl_ref",
    "val_proxy_ratio@10",
    "val_gold_ratio@10",
    "val_gold_reward@10",
    "val_length@10",
    "val_repeat@10",
)

## Trade-off

Every evaluation of every run is a point (step 0 is π_ref, lines follow the steps of a run). Left: relevance against the KL to π_ref; middle: relevance against the gold played ratio of the top-10 (the ratio, not the run's own reward, so that R1 and R4 share an axis); right: gold against proxy ratio, where over-optimisation shows as gold falling behind proxy. Dashed: the reranker over β (the greedy point has no KL, so it shows only in the middle and right panels).

In [ ]:
def plot_tradeoff(runs: list[str], name: str, extra: pl.DataFrame | None = None) -> None:
    """RL runs from their logs; ``extra`` (e.g. the reranker) is drawn dashed."""
    logs = pl.concat(
        [
            pl.read_ndjson(cfg.run_dir.parent / run / "log.jsonl").with_columns(run=pl.lit(run))
            for run in runs
        ]
        + ([] if extra is None else [extra]),
        how="diagonal_relaxed",
    )
    dashed = set() if extra is None else set(extra["run"])
    logs.write_csv(RESULTS / f"{name}.csv")
    axes_spec = [
        ("val_kl_ref", "val_ndcg@10", "KL(π || π_ref)", "NDCG@10"),
        ("val_gold_ratio@10", "val_ndcg@10", "gold ratio@10", "NDCG@10"),
        ("val_proxy_ratio@10", "val_gold_ratio@10", "proxy ratio@10", "gold ratio@10"),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(21, 6))
    for ax, (x, y, xlabel, ylabel) in zip(axes, axes_spec, strict=True):
        for run in [*runs, *sorted(dashed)]:
            log = logs.filter(pl.col("run") == run).sort("step")
            style = {"linestyle": "--", "marker": "x"} if run in dashed else {"marker": "."}
            ax.plot(log[x], log[y], label=run, **style)
        ax.set(xlabel=xlabel, ylabel=ylabel)
    axes[0].set_xscale("symlog", linthresh=0.1)
    axes[0].legend(fontsize=8)
    save_figure(fig, RESULTS / f"{name}.png")
    plt.show()


plot_tradeoff(list(policies), "tradeoff", extra=rerank["val"])

## Final policies vs π_ref (val and test)

The same protocol as stage 2, with and without the repeat filter. The final step of each run is used here; step selection on val (e.g. the best NDCG at a KL budget) uses the logs above.

In [ ]:
COLUMNS = [
    "split",
    "method",
    "exclude_seen",
    "ndcg@10",
    "recall@10",
    "recall@100",
    "paper_ndcg@10",
    "paper_recall@100",
    "coverage@10",
]


base_path = cfg.run_dir.parent / cfg.rl.base_run / "best.pt"
# policies are loaded one at a time; eval sets and MostPop / Repeat are built once
results = evaluate.results_table(
    prepared,
    cfg.data,
    models={"gpt2_base": base_path, **policies},
    amp_dtype=amp_dtype(device, cfg.rl.amp),
    device=device,
)
results.write_csv(RESULTS / "results.csv")
results.select(COLUMNS)